In [2]:
import serpapi
import json
import requests
import pandas as pd
import time

In [3]:
api_key = "cae0d822c2848a3edc91a6fe613ab32ec3ad5f7e3769490a7f5e20c8b84a24ea"
api_key3 = "dc3e23721e31669446e540cc5458ac9a7b9eb9950dbfa04828f00190c66595cb"

### Step 0: Inspect a single raw response FIRST

SerpAPI does not expose a dedicated `ai_overview` for Baidu the way it does for Google.
Any AI-generated or health card content comes back under `answer_box`, `knowledge_graph`,
or occasionally a custom key that varies by query type.

**Run this cell on one query before the full pipeline** to see the exact keys
in your response — then adjust the parsers below if needed.

In [4]:
client = serpapi.Client(api_key=api_key3)

test_response = client.search({
    "engine": "baidu",
    "q": "高血压是什么",  # diabetes — good test case, should trigger a health card
})

print("Top-level keys in Baidu response:")
print(list(test_response.keys()))

print("\n--- Full raw response ---")
print(json.dumps(dict(test_response), indent=2, ensure_ascii=False))

Top-level keys in Baidu response:
['search_metadata', 'search_parameters', 'search_information', 'ai_overview', 'organic_results', 'people_also_search_for', 'related_searches', 'top_searches', 'pagination', 'serpapi_pagination']

--- Full raw response ---
{
  "search_metadata": {
    "id": "6a12172a0e4be0beb1c06d89",
    "status": "Success",
    "json_endpoint": "https://serpapi.com/searches/ZVG4cM8z-IwO_hIiOX182w/6a12172a0e4be0beb1c06d89.json",
    "created_at": "2026-05-23 21:07:54 UTC",
    "processed_at": "2026-05-23 21:07:54 UTC",
    "baidu_url": "https://www.baidu.com/s?wd=%E9%AB%98%E8%A1%80%E5%8E%8B%E6%98%AF%E4%BB%80%E4%B9%88&oq=%E9%AB%98%E8%A1%80%E5%8E%8B%E6%98%AF%E4%BB%80%E4%B9%88&tn=baidu&ie=utf-8&f=8",
    "raw_html_file": "https://serpapi.com/searches/ZVG4cM8z-IwO_hIiOX182w/6a12172a0e4be0beb1c06d89.html",
    "total_time_taken": 4.3
  },
  "search_parameters": {
    "q": "高血压是什么",
    "f": "8",
    "device": "desktop",
    "engine": "baidu"
  },
  "search_information": {
 

In [5]:
# Quick spot-check of the three most likely fields
print("answer_box:",       test_response.get("ai_overview"))
print("\nFirst organic:",   test_response.get("organic_results", [{}])[0])

answer_box: {'text_blocks': [{'type': 'paragraph', 'snippet': '高血压是一种血液在血管内流动时，对血管壁产生的压力持续超出正常范围的慢性疾病。医学上通常将未服用降压药时，非同日三次测量收缩压≥140mmHg 或舒张压≥90mmHg 作为确诊依据 。这种病早期常无明显感觉，因此有“沉默杀手”之称，但若长期不控制，会严重损伤心脏、大脑及肾脏等关键器官 。\u200c\u200c\u200c'}, {'type': 'heading', 'snippet': '🩺 诊断标准与分级'}, {'type': 'list', 'list': [{'snippet': '\u200c确诊流程\u200c：需要在安静状态下，未使用降压药，且非同日进行三次测量，若结果均超标方可确诊 。\u200c\u200c\u200c', 'snippet_highlighted_words': ['确诊流程']}, {'snippet': '\u200c血压分级\u200c：根据升高程度分为三级，级别越高风险越大 。', 'snippet_highlighted_words': ['血压分级'], 'list': [{'snippet': '\u200c1 级高血压\u200c：收缩压 140-159mmHg 或舒张压 90-99mmHg。', 'snippet_highlighted_words': ['1 级高血压']}, {'snippet': '\u200c2 级高血压\u200c：收缩压 160-179mmHg 或舒张压 100-109mmHg。', 'snippet_highlighted_words': ['2 级高血压']}, {'snippet': '\u200c3 级高血压\u200c：收缩压≥180mmHg 或舒张压≥110mmHg。', 'snippet_highlighted_words': ['3 级高血压']}]}, {'snippet': '\u200c测量差异\u200c：诊室血压标准通常为≥140/90mmHg，家庭自测血压标准可能更为严格，如≥135/85mmHg 即需警惕 。部分临床指南建议将控制目标设定在 130/80mmHg 以下，具体需遵医嘱 。\u200c\u200c\u200c7百科\

### Functions

Baidu's `ai_overview` has the same structure as Google's: `text_blocks` and `references`.
Parsers are adapted from the Google notebook.

In [6]:
def query_baidu(search_query: str, api_key: str) -> dict:
    client = serpapi.Client(api_key=api_key)
    results = client.search({
        "engine": "baidu",
        "q": search_query,
    })
    return results

In [7]:
def resolve_refs(indexes: list, ref_lookup: dict) -> list:
    return [
        ref_lookup[i]
        for i in indexes
        if i in ref_lookup
    ]

def parse_ai_overview(ai_overview: dict) -> dict:
    ref_lookup = {
        ref["index"]: {
            "title":  ref.get("title", ""),
            "link":   ref.get("link", ""),
            "source": ref.get("source", "")
        }
        for ref in ai_overview.get("references", [])
    }

    parsed_sections = []

    for block in ai_overview.get("text_blocks", []):
        block_type = block.get("type")

        if block_type == "paragraph":
            parsed_sections.append({
                "type":          "paragraph",
                "content":       block.get("snippet", ""),
                "cited_sources": resolve_refs(block.get("reference_indexes", []), ref_lookup)
            })

        elif block_type == "list":
            items = [item.get("snippet", "") for item in block.get("list", [])]
            parsed_sections.append({
                "type":          "list",
                "items":         items,
                "cited_sources": resolve_refs(block.get("reference_indexes", []), ref_lookup)
            })

    return {
        "sections":       parsed_sections,
        "all_references": list(ref_lookup.values())
    }

In [8]:
def extract_related_searches(serpapi_response: dict) -> list:
    return [
        item.get("query", "")
        for item in serpapi_response.get("related_searches", [])
    ]

### Queries

Each entry: `(english_label, mandarin_query)`  
Merge key with Google dataset: `query_en`

In [9]:
queries = [
    # --- Cancer ---
    ("breast cancer", "乳腺癌"),
    ("what is cancer", "癌症是什么"),
    ("cancer symptoms", "癌症症状"),
    ("lung cancer", "肺癌"),
    ("skin cancer", "皮肤癌"),
    ("colon cancer", "结肠癌"),
    ("cancer treatment", "癌症治疗"),
    ("prostate cancer", "前列腺癌"),
    ("blood cancer", "血癌"),
    ("pancreatic cancer", "胰腺癌"),
    # --- Diabetes ---
    ("diabetes type 2", "2型糖尿病"),
    ("type 2 diabetes", "二型糖尿病"),
    ("what is diabetes", "糖尿病是什么"),
    ("diabetes type 1", "1型糖尿病"),
    ("type 1 diabetes", "一型糖尿病"),
    ("diabetes symptoms", "糖尿病症状"),
    ("gestational diabetes", "妊娠糖尿病"),
    ("insulin", "胰岛素"),
    ("symptoms of diabetes", "糖尿病的症状"),
    ("diabetes test", "糖尿病检测"),
    # --- Hypertension ---
    ("hypertension blood pressure", "高血压"),
    ("blood pressure", "血压"),
    ("pulmonary hypertension", "肺动脉高压"),
    ("what is hypertension", "高血压是什么"),
    ("hypertension symptoms", "高血压症状"),
    ("high blood pressure", "血压高"),
    ("hypertension treatment", "高血压治疗"),
    ("intracranial hypertension", "颅内高压"),
    ("stage 2 hypertension", "二级高血压"),
    ("hypertension stage 2", "高血压二期"),
    # --- Depression ---
    ("anxiety", "焦虑"),
    ("what is depression", "抑郁症是什么"),
    ("anxiety and depression", "焦虑和抑郁"),
    ("depression and anxiety", "抑郁和焦虑"),
    ("depression treatment", "抑郁症治疗"),
    ("depression symptoms", "抑郁症症状"),
    ("depression help", "抑郁症求助"),
    ("postpartum", "产后"),
    ("postpartum depression", "产后抑郁"),
    ("depression medication", "抑郁症药物"),
    # --- Hantavirus ---
    ("the hantavirus", "汉坦病毒"),
    ("virus", "病毒"),
    ("hantavirus symptoms", "汉坦病毒症状"),
    ("virus hantavirus", "汉坦病毒感染"),
    ("hantavirus outbreak", "汉坦病毒爆发"),
    ("virus hantavirus outbreak", "汉坦病毒疫情"),
    ("what is hantavirus", "汉坦病毒是什么"),
    ("hantavirus 2026", "汉坦病毒2026"),
    ("hantavirus cruise", "汉坦病毒邮轮"),
    ("hantavirus update", "汉坦病毒最新消息"),
    # --- Long COVID ---
    ("covid symptoms", "新冠症状"),
    ("long covid symptoms", "新冠后遗症症状"),
    ("what is long covid", "什么是新冠后遗症"),
    ("how long does covid last", "新冠持续多久"),
    ("long term covid", "长期新冠"),
    ("how long was covid", "新冠流行多久了"),
    ("symptoms of covid", "新冠肺炎症状"),
    ("symptoms of long covid", "长新冠症状"),
    ("long covid treatment", "新冠后遗症治疗"),
    ("covid 19", "新冠肺炎"),
    # --- Acupuncture ---
    ("acupuncture near me", "附近针灸"),
    ("what is acupuncture", "针灸是什么"),
    ("acupuncture points", "针灸穴位"),
    ("acupuncture for pain", "针灸治疗疼痛"),
    ("massage", "按摩"),
    ("chinese acupuncture", "中医针灸"),
    ("community acupuncture", "社区针灸"),
    ("does acupuncture work", "针灸有效吗"),
    ("fertility acupuncture", "助孕针灸"),
    ("acupuncture needles", "针灸针"),
    # --- Cupping ---
    ("cupping therapy near me", "附近拔罐"),
    ("cupping near me", "附近拔火罐"),
    ("what is cupping therapy", "拔罐是什么"),
    ("cupping massage therapy", "拔罐推拿"),
    ("massage cupping therapy", "推拿拔罐疗法"),
    ("cupping massage", "拔罐按摩"),
    ("what is cupping", "什么是拔罐"),
    ("massage", "按摩"),
    ("cupping therapy benefits", "拔罐的好处"),
    ("cupping set", "拔罐套装"),
    # --- Herbal medicine ---
    ("herbal chinese medicine", "中草药"),
    ("chinese medicine", "中医"),
    ("chinese herbal medicine", "中药"),
    ("what is herbal medicine", "草药是什么"),
    ("traditional herbal medicine", "传统草药"),
    ("herbal medicine near me", "附近中药店"),
    ("acupuncture", "针灸"),
    ("acupuncture herbal medicine", "针灸中药"),
    ("herbal medicine school", "中医药学校"),
    ("herbal tea", "草药茶"),
    # --- Abortion ---
    ("what is abortion", "什么是人工流产"),
    ("abortion clinic", "人工流产诊所"),
    ("abortion laws", "人工流产法律"),
    ("planned parenthood", "计划生育"),
    ("planned parenthood abortion", "计划生育手术"),
    ("medical abortion", "药物流产"),
    ("what is an abortion", "人工流产是什么"),
    ("supreme court abortion", "最高法院堕胎裁决"),
    ("abortion pill online", "网上购买堕胎药"),
    ("abortion meaning", "流产的含义"),
    # --- HIV ---
    ("the hiv", "艾滋病毒"),
    ("what hiv", "什么是艾滋"),
    ("what is hiv", "艾滋病是什么"),
    ("aids", "艾滋病"),
    ("hiv aids", "艾滋病毒感染"),
    ("hiv symptoms", "艾滋病症状"),
    ("hiv test", "艾滋病检测"),
    ("hiv positive", "艾滋病阳性"),
    ("hiv treatment", "艾滋病治疗"),
    ("hiv testing", "HIV检测"),
    # --- Gender-affirming care ---
    ("gender affirming care", "性别肯定医疗"),
    ("gender affirming surgery", "变性手术"),
    ("what is gender affirming care", "什么是性别肯定医疗"),
    ("gender affirming care definition", "性别肯定治疗定义"),
    ("gender affirming care near me", "附近跨性别医疗"),
    ("gender affirming surgery", "性别重置手术"),
    ("gender reassignment surgery", "性别转换手术"),
    ("gender reassignment surgery male to female", "男变女手术"),
    ("how does gender reassignment surgery work", "变性手术如何进行"),
    ("gender affirming hormone therapy", "跨性别激素治疗"),
]

### Pipeline

In [12]:
def run_pipeline_for_queries(queries, api_key) -> pd.DataFrame:
    rows = []
    total = len(queries)

    for i, (english_label, mandarin_query) in enumerate(queries):
        print(f"[{i+1}/{total}] EN: '{english_label}' | ZH: '{mandarin_query}'")

        try:
            results = query_baidu(mandarin_query, api_key)

            ai_overview_raw = results.get("ai_overview")
            related         = extract_related_searches(results)

            if not ai_overview_raw:
                rows.append({
                    "query_en":       english_label,
                    "query_zh":       mandarin_query,
                    "has_ai_overview": False,
                    "all_items":       None,
                    "cited_sources":   None,
                    "all_references":  None,
                    "related_searches": related,
                })
                continue

            parsed     = parse_ai_overview(ai_overview_raw)
            all_items  = []
            all_sources = []

            for section in parsed["sections"]:
                if section["type"] == "list":
                    all_items.extend(section.get("items", []))
                all_sources.extend(section.get("cited_sources", []))

            # Deduplicate sources
            seen = set()
            unique_sources = []
            for s in all_sources:
                if s["link"] not in seen:
                    seen.add(s["link"])
                    unique_sources.append(s)

            rows.append({
                "query_en":        english_label,
                "query_zh":        mandarin_query,
                "has_ai_overview": True,
                "all_items":       all_items,
                "cited_sources":   unique_sources,
                "all_references":  parsed["all_references"],
                "related_searches": related,
            })

        except Exception as e:
            print(f"   ERROR on '{mandarin_query}': {e}")
            rows.append({
                "query_en":        english_label,
                "query_zh":        mandarin_query,
                "has_ai_overview": False,
                "all_items":       None,
                "cited_sources":   None,
                "all_references":  None,
                "related_searches": None,
            })

        time.sleep(1.5)

    return pd.DataFrame(rows)

In [13]:
df = run_pipeline_for_queries(queries, api_key3)
df

[1/120] EN: 'breast cancer' | ZH: '乳腺癌'
[2/120] EN: 'what is cancer' | ZH: '癌症是什么'
[3/120] EN: 'cancer symptoms' | ZH: '癌症症状'
[4/120] EN: 'lung cancer' | ZH: '肺癌'
[5/120] EN: 'skin cancer' | ZH: '皮肤癌'
[6/120] EN: 'colon cancer' | ZH: '结肠癌'
[7/120] EN: 'cancer treatment' | ZH: '癌症治疗'
[8/120] EN: 'prostate cancer' | ZH: '前列腺癌'
[9/120] EN: 'blood cancer' | ZH: '血癌'
[10/120] EN: 'pancreatic cancer' | ZH: '胰腺癌'
[11/120] EN: 'diabetes type 2' | ZH: '2型糖尿病'
[12/120] EN: 'type 2 diabetes' | ZH: '二型糖尿病'
[13/120] EN: 'what is diabetes' | ZH: '糖尿病是什么'
[14/120] EN: 'diabetes type 1' | ZH: '1型糖尿病'
[15/120] EN: 'type 1 diabetes' | ZH: '一型糖尿病'
[16/120] EN: 'diabetes symptoms' | ZH: '糖尿病症状'
[17/120] EN: 'gestational diabetes' | ZH: '妊娠糖尿病'
[18/120] EN: 'insulin' | ZH: '胰岛素'
[19/120] EN: 'symptoms of diabetes' | ZH: '糖尿病的症状'
[20/120] EN: 'diabetes test' | ZH: '糖尿病检测'
[21/120] EN: 'hypertension blood pressure' | ZH: '高血压'
[22/120] EN: 'blood pressure' | ZH: '血压'
[23/120] EN: 'pulmonary hypertension' | Z

,query_en,query_zh,has_ai_overview,all_items,cited_sources,all_references,related_searches
0,breast cancer,乳腺癌,True,"[‌乳房肿块‌：最常见，多为无痛、质地硬、边缘不规则的肿块，活动度差。, ‌皮肤改变‌：皮肤...",[],"[{'title': '乳腺癌', 'link': 'https://www.youlai....","[乳腺瘤一般能活几年, 乳腺瘤是什么引起的, 全乳切除多久能自理, 人乳头瘤早期图片, 乳腺..."
1,what is cancer,癌症是什么,True,"[‌基因突变‌：细胞DNA损伤累积，导致原癌基因激活和抑癌基因失活, ‌信号紊乱‌：细胞接收...",[],[],"[靶向针多少钱一针, 肿瘤的样子图片, 肿瘤切开了里面是什么, 肿瘤一般能活多久, 什么是肿..."
2,cancer symptoms,癌症症状,True,[‌摸到无痛硬块‌：身体任何部位出现质地硬、边界不清、推不动的肿块，像乳房或淋巴结肿大，别以...,[],"[{'title': '癌症的症状是怎样的', 'link': 'https://m.boh...","[靶向针多少钱一针, 肿瘤的样子图片, 身体哪种信号是肿瘤, 怎么知道自己长肿瘤了, 癌早期..."
3,lung cancer,肺癌,True,[‌老烟民和吸二手烟的人‌：长期吸烟是头号凶手，烟草里有几十种致癌物，直接损伤肺部细胞，吸烟...,"[{'title': '肺癌', 'link': 'https://baike.baidu....","[{'title': '肺癌', 'link': 'https://baike.baidu....","[靶向针多少钱一针, 70岁老人肺部肿瘤能活多久, 肺部肿瘤早期能活多久, 靶向药2025价..."
4,skin cancer,皮肤癌,True,[‌痣或斑块突然变样‌：原有的痣若出现‌不对称‌、‌边缘不规则‌、‌颜色不均匀‌、‌直径超过...,[],"[{'title': '确诊5个月便离世!这种痣,可能要命', 'link': 'https...","[白癜风怎么治疗, 如何判断是不是白癜风, 100种皮肤病照片, 扫一扫皮肤病识别, 皮肤肿..."
...,...,...,...,...,...,...,...
115,gender affirming surgery,性别重置手术,True,"[‌术前评估‌, ‌手术类型‌, ‌身体风险‌, ‌心理与社会适应‌, ‌身体恢复‌, ‌功...",[],[],"[变性一年后外阴效果图, 去势手术, 男生被绑在手术台上做变性手术二次, 40多可以做变性手..."
116,gender reassignment surgery,性别转换手术,False,None,None,None,"[性别转换生成器, 一觉醒来我就变成了女孩子, 男变女手术, 变性一年后外阴效果图, 系统将..."
117,gender reassignment surgery male to female,男变女手术,True,[‌心理评估与法律程序‌ 需接受至少12个月的心理评估，由精神科医生确认性别焦虑症诊断，并开...,[],[],"[男性割掉下体手术, 男科手术视频(真人), 女性外阴切除手术实拍, 变性一年后外阴效果图,..."
118,how does gender reassignment surgery work,变性手术如何进行,False,None,None,None,"[男变性手术, 做变性手术需要多少钱, 女的变男的变性手术是怎么变, 怎样变性(不用手术),..."


### Save

In [14]:
file_name = "baidu.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")
print("Saved to", file_name)

Saved to baidu.csv
